In [1]:
import os

In [2]:
%pwd

'd:\\PredictBot-Score-MLOps\\research'

In [3]:
os.chdir('..')

In [4]:
%pwd

'd:\\PredictBot-Score-MLOps'

In [5]:
from dataclasses import dataclass
from pathlib import Path
from src.predictor_bot_score.config.configuration import yaml_load , create_directories
from src.predictor_bot_score.logger import logger
from src.predictor_bot_score.constants import CONFIG_PATH
from src.predictor_bot_score.utils.model_factory import get_model , get_fit_kwargs , get_mlflow_logger , MODEL_REGISTRY
from sklearn.metrics import mean_absolute_error
import os
import numpy as np
import pandas as pd
import glob
import mlflow
import lightgbm
import pickle
from datetime import datetime
import sqlite3
import gc


In [6]:
@dataclass(frozen=True)
class ModelTrainingConfig:
    train_data_path       : Path
    val_data_path         : Path
    model_dir             : Path
    active_model_strategy : str
    models                : dict
    features              : list[str]
    target_column         : str
    baseline_mae          : float
    promotion_criteria    : dict
    mlflow_experiment     : str
    mlflow_tracking_uri   : str

In [7]:
class config_manager:

    def __init__(self, config = CONFIG_PATH):

        self.config = yaml_load(config)
        
        create_directories([self.config.artifacts_root])

    def get_model_training_config(self) -> ModelTrainingConfig:

        config = self.config.model_training

        create_directories([config.model_dir])

        return ModelTrainingConfig(
        train_data_path       = Path(config.train_data_path),
        val_data_path         = Path(config.val_data_path),
        model_dir             = Path(config.model_dir),
        active_model_strategy = config.active_model_strategy,
        models                = dict(config.models),
        features              = list(config.features),
        target_column         = config.target_column,
        baseline_mae          = float(config.baseline_mae),
        promotion_criteria    = dict(config.promotion_criteria),
        mlflow_experiment     = config.mlflow.experiment_name,
        mlflow_tracking_uri   = config.mlflow.tracking_uri,
    )
        

In [ ]:


class Model_Building :

    def __init__(self, config : ModelTrainingConfig):
        self.config = config
        self.train , self.val = self._read_data()

    def _get_files(self , folder:Path ,prefix :str):
        files = glob.glob(os.path.join(folder ,f"{prefix}_*.csv"))
        if not files:
            raise FileNotFoundError(f"No {prefix} file found in {folder}")
        return max(files , key=os.path.getmtime)
    

    def _read_data(self):
        try:
            logger.info("=" * 50)
            logger.info("Reading train /  test splits")
            logger.info("=" * 50)

            train_file = self._get_files(self.config.train_data_path, "train")
            val_file   = self._get_files(self.config.val_data_path,   "val")

            train = pd.read_csv(train_file)
            val   = pd.read_csv(val_file)
            
            logger.info(f"Train : {len(train)} rows")
            logger.info(f"Val   : {len(val)} rows")

            return train, val

        except FileNotFoundError as e:
            logger.error(f"Split file not found: {e}")
            raise

        except Exception as e:
            logger.error(f"Failed to read splits: {str(e)}")
            raise

    def prepare_data(self ,df :pd.DataFrame):
        try:
            X = df[self.config.features]
            y = df[self.config.target_column]

            return X ,y
        except Exception as e:
            raise e 
        
    def _smape(self, actual , predicted ):
        try:
            sampe_result = float(
                100 * np.mean(
                    2 * np.abs(predicted - actual) /
                    (np.abs(actual) + np.abs(predicted) + 1e-8)
                )
            )

            return sampe_result
        except Exception as e:
            logger.error(f"SMAPE calculation failed: {str(e)}")
            raise 
    
    def model_training(self):
        try:
            X_train, y_train = self.prepare_data(self.train)
            X_val,   y_val   = self.prepare_data(self.val)

            trained_models = {}

            for model_name, cfg in self.config.models.items():
                model_type   = cfg["type"]
                model_params = cfg["params"]

                if model_type not in MODEL_REGISTRY:
                    logger.warning(
                        f"Skipping {model_name}: '{model_type}' not in registry"
                    )
                    continue

                logger.info("")
                logger.info(f"--- Training {model_name} ({model_type}) ---")
                logger.info("-" * 50)

                model      = get_model(model_type, model_params)
                eval_set   = [(X_val, y_val)]
                fit_kwargs = get_fit_kwargs(model_type, eval_set)

                model.fit(X_train, y_train, **fit_kwargs)

                val_pred  = model.predict(X_val)
                val_mae   = float(mean_absolute_error(y_val, val_pred))
                val_smape = self._smape(y_val.values, val_pred)

                trained_models[model_name] = {
                    "model"      : model,
                    "model_type" : model_type,
                    "params"     : model_params,
                    "val_mae"    : val_mae,
                    "val_smape"  : val_smape,
                }

                logger.info(f"{model_name} VAL MAE   : {val_mae}")
                logger.info(f"{model_name} VAL SMAPE : {val_smape}")
                logger.info(f"PASSED - {model_name} trained")

            return trained_models

        except Exception as e:
            logger.error(f"Training failed: {str(e)}")
            raise
    
    def save_model(self,model , model_name):

        try:
            path = self.config.model_dir
            time_stamp = datetime.now().strftime("%Y_%m_%d_%H")
            

            run_dir = os.path.join(path ,f'run__{time_stamp}')
            os.makedirs(run_dir , exist_ok=True)

            model_path = os.path.join(run_dir,f"{model_name}__.pkl")
            
            with open(model_path ,"wb") as f:
                pickle.dump(model , f)
                f.close()
        
            logger.info(f"Model saved  {model_path}")
            logger.info("PASSED - Model saved")
            logger.info("-" * 50)

            return model_path

        except Exception as e:
            logger.error(f"Failed to save model: {str(e)}")
            raise

    # ── log to mlflow ─────────────────────────────────────
    def log_to_mlflow(self, model_name, result: dict, model_path: str):
        try:
            logger.info("")
            logger.info(f"fLOGGING {model_name} TO MLFLOW")
            logger.info("-" * 50)

            mlflow.set_tracking_uri(self.config.mlflow_tracking_uri)
            mlflow.set_experiment(self.config.mlflow_experiment)

            with mlflow.start_run(run_name=model_name):

                mlflow.log_params(result['params'])

                mlflow.log_metric("val_mae",   result["val_mae"])
                mlflow.log_metric("val_smape", result["val_smape"])

                # tags
                mlflow.set_tag("model_type",  result["model_type"])
                mlflow.set_tag("stage",       "Staging")
                mlflow.set_tag("trained_at",  datetime.now().isoformat())
                mlflow.set_tag("model_path",  model_path)
                mlflow.set_tag("model_name", model_name)      # new
                mlflow.set_tag("batch_id", self.batch_id) 

                model_log = get_mlflow_logger(result['model_type'])
                model_log(result['model'],artifact_path="model")
                
            logger.info("PASSED - MLflow logging complete")
            logger.info("-" * 50)

        except Exception as e:
            logger.error(f"MLflow logging failed: {str(e)}")
            raise

    def run(self):
        trained_models = None

        try:
            logger.info("=" * 50)
            logger.info(f'{"=" * 50},MODEL TRAINING PIPELINE STARTED')
            logger.info("=" * 50)

            trained_models = self.model_training()

            logger.info("")
            logger.info("=" * 50)
            logger.info("TRAINING SUMMARY")
            logger.info("=" * 50)
            for name, res in trained_models.items():
                logger.info(
                    f"{name} MAE: {res['val_mae']}  "
                    f"SMAPE: {res['val_smape']}"
                )

            logger.info("")
            logger.info("=" * 50)
            logger.info("SAVING AND LOGGING ALL MODELS")
            logger.info("=" * 50)

            for model_name, result in trained_models.items():
                
                model_path = self.save_model(result["model"], model_name) ## this function creates the model which acts as the input for mext self.log_mlflow
                
                self.log_to_mlflow(model_name, result, model_path)

            logger.info("")
            logger.info("=" * 50)
            logger.info("MODEL TRAINING PIPELINE COMPLETE")
            logger.info("=" * 50)

            return None

        except Exception as e:
            logger.error(f"Model training pipeline failed: {str(e)}")
            raise

        finally:
            if trained_models is not None:
                for res in trained_models.values():
                    res["model"] = None
                trained_models = None

            self.train = None
            self.val   = None
            gc.collect()
            logger.info("Memory cleared")
            logger.info(f'{"=" * 50},MODEL TRAINING PIPELINE COMPLETED')



             

   



            

   

In [15]:
xc = config_manager()
xc = xc.get_model_training_config()
xc = Model_Building(xc)
xc.run()


[2026-06-24 21:10:55,317: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-24 21:10:55,323: INFO: common: Directory created (or already exists) at: artifacts]
[2026-06-24 21:10:55,325: INFO: common: Directory created (or already exists) at: artifacts/Model_Building]
[2026-06-24 21:10:55,327: INFO: 846615201: ==================================================]
[2026-06-24 21:10:55,330: INFO: 846615201: Reading train /  test splits]
[2026-06-24 21:10:55,330: INFO: 846615201: ==================================================]
[2026-06-24 21:10:55,490: INFO: 846615201: Train : 28426 rows]
[2026-06-24 21:10:55,492: INFO: 846615201: Val   : 4061 rows]
[2026-06-24 21:10:55,494: INFO: 846615201: ==================================================]
[2026-06-24 21:10:55,496: INFO: 846615201: ==================================================,MODEL TRAINING PIPELINE STARTED]
[2026-06-24 21:10:55,498: INFO: 846615201: ==================================================]
[20

2026/06/24 21:11:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/24 21:11:24 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


[2026-06-24 21:11:38,683: INFO: 846615201: PASSED - MLflow logging complete]
[2026-06-24 21:11:38,683: INFO: 846615201: --------------------------------------------------]
[2026-06-24 21:11:38,813: INFO: 846615201: Model saved  artifacts\Model_Building\run__2026_06_24_21\xgboost_v1__.pkl]
[2026-06-24 21:11:38,816: INFO: 846615201: PASSED - Model saved]
[2026-06-24 21:11:38,816: INFO: 846615201: --------------------------------------------------]
[2026-06-24 21:11:38,818: INFO: 846615201: ]
[2026-06-24 21:11:38,820: INFO: 846615201: fLOGGING xgboost_v1 TO MLFLOW]
[2026-06-24 21:11:38,820: INFO: 846615201: --------------------------------------------------]


2026/06/24 21:11:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


[2026-06-24 21:11:50,315: INFO: 846615201: PASSED - MLflow logging complete]
[2026-06-24 21:11:50,318: INFO: 846615201: --------------------------------------------------]
[2026-06-24 21:11:50,318: INFO: 846615201: ]
[2026-06-24 21:11:50,321: INFO: 846615201: ==================================================]
[2026-06-24 21:11:50,322: INFO: 846615201: MODEL TRAINING PIPELINE COMPLETE]
[2026-06-24 21:11:50,325: INFO: 846615201: ==================================================]
[2026-06-24 21:11:50,960: INFO: 846615201: Memory cleared]
[2026-06-24 21:11:50,960: INFO: 846615201: ==================================================,MODEL TRAINING PIPELINE COMPLETED]


In [26]:
yaml = yaml_load(Path('config\config.yaml'))

[2026-06-18 12:05:00,003: INFO: common: yaml file: config\config.yaml loaded successfully]


In [31]:
yaml.model_training.mlflow.experiment_name

'predict-bot-training'